# 13주 · 모비율 검정·원본 ANOVA 자료 대조

공학통계학 CE202

**목표**: 자료·확률·분포·추정·검정의 원리 이해와 통계적 해석 심화

학습 주제: 가설검정 (3) / 분산분석

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week13/CE202_W13.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week13)

질문·가정 확인 → 손계산·예측 → 계산·시각화 → 한 조건 변경 → 통계적 해석·한계 설명

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 기준 결과와 조건을 바꾼 결과를 비교

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE202-statistics-for-engineers/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


<a id="s01"></a>

## 배터리 500개 중 5개 불량

**먼저 예측**: 검정통계량의 표준오차는 p0와 표본비율 중 무엇을 쓰는가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
x,n,p0=5,500,.02
z_prop=(x/n-p0)/np.sqrt(p0*(1-p0)/n)
p_normal=stats.norm.cdf(z_prop); p_exact=stats.binomtest(x,n,p=p0,alternative="less").pvalue
print({"귀무가설 기준 z":z_prop,"정규근사 p":p_normal,"정확 이항 p":p_exact})
assert np.isclose(p_exact,stats.binom.cdf(5,500,.02))
assert p_exact>.05 and p_normal>.05

**조건 변경**: 비율·표본수를 줄인 희소사건 사본에서 근사와 정확검정 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: p0=.02·x5·n500·왼쪽 단측·α.05와 근사 조건 명시

**결과 검토**: 신뢰구간과 검정에서 표준오차 기준이 다를 수 있음

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


<a id="s02"></a>

## 슬라이드·jamovi와 일치하는 기계 6대

**먼저 예측**: F가 작으면 모든 기계 평균이 같다는 사실이 증명되는가?

계산 전에 분모·단위·조건을 한 줄로 기록


In [ ]:
machine_groups={'1': [17.5, 16.9, 15.8, 18.6], '2': [16.4, 19.2, 17.7, 15.4], '3': [20.3, 15.7, 17.8, 18.9], '4': [14.6, 16.7, 20.8, 18.9], '5': [17.5, 19.2, 16.5, 20.5], '6': [18.3, 16.2, 17.5, 20.1]}
groups=[np.array(v) for v in machine_groups.values()]
anova=anova_summary(groups); scipy_anova=stats.f_oneway(*groups,equal_var=True)
print(pd.DataFrame({"machine":list(machine_groups),"n":[len(g) for g in groups],"mean":[g.mean() for g in groups],"sample_variance":[g.var(ddof=1) for g in groups]}))
print(anova)
assert np.isclose(anova["F"],.3068007662835247) and np.isclose(anova["p"],.9024205151050708)
assert np.isclose(anova["F"],scipy_anova.statistic)
print("Welch 대안(다른 모형):",stats.f_oneway(*groups,equal_var=False))
plt.boxplot(groups,tick_labels=["M1","M2","M3","M4","M5","M6"]);plt.ylabel("Strength (source unit unspecified)");plt.show()

**조건 변경**: 기계6을 기계5 값으로 복사한 원본 Excel 입력 사본과 F·p 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 기준 셀은 보존

**선택 심화 · AI 검토**: 6집단×4·등분산 ANOVA·자유도5/18·고전/Welch 구분·인장강도 단위 확인 명시

**결과 검토**: 슬라이드18·jamovi·Excel 저장 결과에 일치하는 정리본 사용. 원본 불일치는 DATA_RECONCILIATION.md 참고

예측·손계산·변경 결과·개념 설명·적용 한계 기록


In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

원본 ANOVA 결과와 기각 못함의 해석 기록

입력 항목:

- `anova_F`: 기계 6대 정리본의 고전 등분산 ANOVA F · 수치
- `anova_df_between`: 기계 6대 ANOVA의 집단간 자유도 · 수치
- `anova_df_within`: 기계 6대 ANOVA의 집단내 자유도 · 수치
- `non_rejection_proves_all_means_equal`: ANOVA 기각 못함이 모든 모평균의 동일성 입증인가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토


In [ ]:
answers = {'anova_F': None, 'anova_df_between': None, 'anova_df_within': None, 'non_rejection_proves_all_means_equal': None}


In [ ]:
def check_answers(values):
    expected = {'anova_F': 0.3068007662835247, 'anova_df_between': 5, 'anova_df_within': 18, 'non_rejection_proves_all_means_equal': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
